In [ ]:
import warnings
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.widgets import Slider
import pickle
from scipy.stats import beta

warnings.filterwarnings("ignore")

LINE_WIDTH = 2
%matplotlib tk

# Subfuntions

In [2]:
def sum_by_refpos(data):
    if not data:
        return np.array([], dtype=np.float32)

    values = np.array([item[0] for item in data], dtype=np.float32)
    refs = np.array([item[1][3] for item in data], dtype=np.int8)

    max_ref = int(np.max(refs))

    sums = np.bincount(refs, weights=values, minlength=max_ref + 1).astype(np.float32)

    counts = np.bincount(refs, minlength=max_ref + 1)

    means = np.divide(
        sums, counts, out=np.zeros_like(sums, dtype=np.float32), where=counts > 0
    )

    return means


def pol2cart(rho, phi, z=None):
    x = rho * np.cos(phi)
    y = rho * np.sin(phi)
    if z is not None:
        return x, y, z
    return x, y


ALPHA = 0.15

# Data loading

In [3]:
with open("y_data_3_small.pkl", "rb") as file:
    y_data_3_small = pickle.load(file)

with open("y_data_4_small.pkl", "rb") as file:
    y_data_4_small = pickle.load(file)

with open("y_data_5_small.pkl", "rb") as file:
    y_data_5_small = pickle.load(file)

with open("y_data_6_small.pkl", "rb") as file:
    y_data_6_small = pickle.load(file)

with open("y_data_7_small.pkl", "rb") as file:
    y_data_7_small = pickle.load(file)

with open("y_data_8_small.pkl", "rb") as file:
    y_data_8_small = pickle.load(file)

with open("y_data_mov_3_small.pkl", "rb") as file:
    y_data_mov_3_small = pickle.load(file)

with open("y_data_mov_4_small.pkl", "rb") as file:
    y_data_mov_4_small = pickle.load(file)

with open("y_data_mov_5_small.pkl", "rb") as file:
    y_data_mov_5_small = pickle.load(file)

with open("y_data_3.pkl", "rb") as file:
    y_data_3_big = pickle.load(file)

with open("y_data_4.pkl", "rb") as file:
    y_data_4_big = pickle.load(file)

with open("y_data_mov_3.pkl", "rb") as file:
    y_data_mov_3_big = pickle.load(file)

with open("y_data_mov_4.pkl", "rb") as file:
    y_data_mov_4_big = pickle.load(file)

with open("y_data_mov_5.pkl", "rb") as file:
    y_data_5_big = pickle.load(file)

with open("y_data_mov_6.pkl", "rb") as file:
    y_data_6_big = pickle.load(file)

# Electrodes positions

## Body parameters

In [6]:
R_CHEST = 0.35
R_HIPS = 0.4
R_BELLY = 0.1
A = 8.0
C = 6.0
V0 = -0.1
W = 0.3

u = np.linspace(0, 2 * np.pi, 120)
v = np.linspace(-0.5, 0.5, 120)
u, v = np.meshgrid(u, v)

R_base = R_CHEST * np.exp(-A * (v - 0.3) ** 2) + R_HIPS * np.exp(-C * (v + 0.3) ** 2)

belly_profile = 1 / (1 + ((v - V0) / W) ** 4)
front = np.maximum(0, np.cos(u))

belly = R_BELLY * belly_profile * front

R = R_base + belly

x = R * np.cos(u)
x += 0.1 * np.exp(-10 * (v - V0) ** 2)
y = R * np.sin(u)
z = v

## Hearts parameters

In [7]:
N_PHI_HEARTS_LEFT = 7
N_PHI_HEARTS_RIGHT = 4
N_Z_HEARTS = 4
N_R = 2
phi_left_hearts = np.linspace(0, 5 * np.pi / 18, N_PHI_HEARTS_LEFT)
phi_right_hearts = np.linspace(-np.pi / 17, -np.pi / 4, N_PHI_HEARTS_RIGHT)
phi_hearts = np.concatenate([phi_left_hearts, phi_right_hearts])
z_low_hearts = np.linspace(-0.35, -0.07, N_Z_HEARTS)
z_high_hearts = np.array([0, 0.025, 0.04])
z_lin_hearts = np.concatenate([z_low_hearts, z_high_hearts])

d_vals = np.linspace(0.04, 0.11, N_R)
phi_3d, z_3d, d_3d = np.meshgrid(phi_hearts, z_lin_hearts, d_vals, indexing="ij")
R_base_pts = R_CHEST * np.exp(-A * (z_3d - 0.3) ** 2) + R_HIPS * np.exp(
    -C * (z_3d + 0.3) ** 2
)

belly_profile_pts = 1 / (1 + ((z_3d - V0) / W) ** 4)
front_pts = np.maximum(0, np.cos(phi_3d))
belly_pts = R_BELLY * belly_profile_pts * front_pts

R_surface = R_base_pts + belly_pts

R_inner = R_surface - d_3d

R_inner = np.maximum(R_inner, 0)

x_center = 0.1 * np.exp(-10 * (z_3d - V0) ** 2)

x_inner = x_center + R_inner * np.cos(phi_3d)
y_inner = R_inner * np.sin(phi_3d)
z_inner = z_3d
phi_flat = phi_3d.flatten()
R_flat = R_inner.flatten()
z_flat = z_3d.flatten()
points_inner_cyl = np.column_stack((phi_flat, R_flat, z_flat))

## Electrodes parameters

In [8]:
params_small = {
    "N_PHI_PTS_LEFT": 4,
    "N_PHI_PTS_RIGHT": 3,
    "N_Z_PTS": 2,
    "z_low_pts": np.linspace(-0.4, -0.25, 2),
    "z_high_pts": np.array([-0.1, 0.05]),
}

params_big = {
    "N_PHI_PTS_LEFT": 6,
    "N_PHI_PTS_RIGHT": 4,
    "N_Z_PTS": 3,
    "z_low_pts": np.linspace(-0.4, -0.1, 3),
    "z_high_pts": np.array([0, 0.1]),
}


def compute_points_cyl(
    N_PHI_PTS_LEFT,
    N_PHI_PTS_RIGHT,
    N_Z_PTS,
    z_low_pts,
    z_high_pts,
    R_CHEST,
    A,
    R_HIPS,
    C,
    V0,
    W,
    R_BELLY,
):
    z_lin_pts = np.concatenate([z_low_pts, z_high_pts])

    phi_left_pts = np.linspace(0, np.pi / 3, N_PHI_PTS_LEFT)
    phi_right_pts = np.linspace(-np.pi / 3.5, -np.pi / 12, N_PHI_PTS_RIGHT)
    phi_pts = np.concatenate([phi_left_pts, phi_right_pts])

    phi_pts, z_lin_pts = np.meshgrid(phi_pts, z_lin_pts)

    R_base_pts = R_CHEST * np.exp(-A * (z_lin_pts - 0.3) ** 2) + R_HIPS * np.exp(
        -C * (z_lin_pts + 0.3) ** 2
    )

    belly_profile_pts = 1 / (1 + ((z_lin_pts - V0) / W) ** 4)
    front_pts = np.maximum(0, np.cos(phi_pts))
    belly_pts = R_BELLY * belly_profile_pts * front_pts

    R_pts = R_base_pts + belly_pts

    x_pts = R_pts * np.cos(phi_pts)
    x_pts += 0.1 * np.exp(-10 * (z_lin_pts - V0) ** 2)
    y_pts = R_pts * np.sin(phi_pts)

    R_true = np.sqrt(x_pts**2 + y_pts**2)
    phi_true = np.arctan2(y_pts, x_pts)

    phi_flat = phi_true.flatten()
    r_flat = R_true.flatten()
    z_flat = z_lin_pts.flatten()
    points_cyl = np.column_stack((phi_flat, r_flat, z_flat))

    return points_cyl


points_cyl_small = compute_points_cyl(
    **params_small,
    R_CHEST=R_CHEST,
    A=A,
    R_HIPS=R_HIPS,
    C=C,
    V0=V0,
    W=W,
    R_BELLY=R_BELLY,
)

points_cyl_big = compute_points_cyl(
    **params_big, R_CHEST=R_CHEST, A=A, R_HIPS=R_HIPS, C=C, V0=V0, W=W, R_BELLY=R_BELLY
)

## Visualization

### Leg reference

In [27]:
fig_ref = plt.figure(figsize=(8, 8))
ax_ref = fig_ref.add_subplot(111, projection="3d")
ax_ref.plot_surface(x, y, z, alpha=ALPHA)
ax_ref.set_title("Электродная сетка с референсом на ноге")
ax_ref.scatter(x_inner, y_inner, z_inner, s=7, color="k")

refpos = [points_cyl_big[5][0], points_cyl_big[5][1], -0.7]

best_ids = [2, 7, 32, 37]
x_big_best, y_big_best, z_big_best = pol2cart(
    points_cyl_big[best_ids].T[1],
    points_cyl_big[best_ids].T[0],
    points_cyl_big[best_ids].T[2],
)
ax_ref.scatter(x_big_best, y_big_best, z_big_best, s=40, color="g")

x_refpos, y_refpos, z_refpos = pol2cart(refpos[1], refpos[0], refpos[2])
ax_ref.scatter(x_refpos, y_refpos, z_refpos, s=20)
ax_ref.text(x_refpos, y_refpos, z_refpos, "R")
x_big, y_big, z_big = pol2cart(
    points_cyl_big.T[1], points_cyl_big.T[0], points_cyl_big.T[2]
)
ax_ref.scatter(x_big, y_big, z_big, s=10, color="r")


ax_ref.view_init(elev=0, azim=0, roll=0)

### Belly reference

In [14]:
fig_belly = plt.figure(figsize=(12, 6))
ax1 = fig_belly.add_subplot(121, projection="3d")
ax1.plot_surface(x, y, z, alpha=ALPHA)
ax1.set_title(f"Электродная сетка до сокращения, N = {len(points_cyl_big)}")
ax1.scatter(x_inner, y_inner, z_inner, s=7, color="k")
x_big, y_big, z_big = pol2cart(
    points_cyl_big.T[1], points_cyl_big.T[0], points_cyl_big.T[2]
)
ax1.scatter(x_big, y_big, z_big, s=10, color="r")


ax2 = fig_belly.add_subplot(122, projection="3d")
ax2.plot_surface(x, y, z, alpha=ALPHA)
ax2.set_title(f"Электродная сетка после сокращения, N = {len(points_cyl_small)}")
ax2.scatter(x_inner, y_inner, z_inner, s=7, color="k")

x_small, y_small, z_small = pol2cart(
    points_cyl_small.T[1], points_cyl_small.T[0], points_cyl_small.T[2]
)
ax2.scatter(x_small, y_small, z_small, s=10, color="r")

ax1.view_init(elev=0, azim=0, roll=0)
ax2.view_init(elev=0, azim=0, roll=0)
plt.ion()
plt.show()

# Reference electrodes heatmap

In [15]:
heatmap_big = sum_by_refpos(y_data_mov_4_big)
heatmap_small = sum_by_refpos(y_data_mov_4_small)

fig_heatmap = plt.figure(figsize=(12, 8))
ax1_heatmap = fig_heatmap.add_subplot(121, projection="3d")
ax1_heatmap.plot_surface(x, y, z, alpha=ALPHA)
ax1_heatmap.set_title(f"Тепловая карта до сокращения сетки, N = {len(points_cyl_big)}")
# ax1_heatmap.scatter(x_inner, y_inner, z_inner, s=7, color="k")
scatter_heatmap_big = ax1_heatmap.scatter(
    x_big, y_big, z_big, s=10, c=heatmap_big, cmap="magma_r"
)
cbar_big = fig_heatmap.colorbar(scatter_heatmap_big, location="bottom")
cbar_big.set_label("Intensity Scale")


ax2_heatmap = fig_heatmap.add_subplot(122, projection="3d")
ax2_heatmap.plot_surface(x, y, z, alpha=ALPHA)
ax2_heatmap.set_title(
    f"Тепловая карта после до сокращения сетки, N = {len(points_cyl_small)}"
)
# ax2_heatmap.scatter(x_inner, y_inner, z_inner, s=7, color="k")
scatter_heatmap_small = ax2_heatmap.scatter(
    x_small, y_small, z_small, s=10, c=heatmap_small, cmap="magma_r"
)
cbar_small = fig_heatmap.colorbar(scatter_heatmap_small, location="bottom")
cbar_small.set_label("Intensity Scale")
ax1_heatmap.view_init(elev=0, azim=0, roll=0)
ax2_heatmap.view_init(elev=0, azim=0, roll=0)
plt.ion()
plt.show()

# Results

## 3 electrodes (2 active, 1 reference)

In [19]:
y_data_3_big_plot = []
comb_ref_3_big = []
for i in range(len(y_data_3_big)):
    y_data_3_big_plot.append(y_data_3_big[i][0].item())
    comb_ref_3_big.append(y_data_3_big[i][1])
y_data_3_big_plot = np.array(y_data_3_big_plot, dtype=np.float32)
mask_3_big = y_data_3_big_plot > 0.42
plot_combs_3_big = np.array(comb_ref_3_big, dtype=np.int8)[mask_3_big]

y_data_3_small_plot = []
comb_ref_3_small = []
for i in range(len(y_data_3_small)):
    y_data_3_small_plot.append(y_data_3_small[i][0].item())
    comb_ref_3_small.append(y_data_3_small[i][1])
y_data_3_small_plot = np.array(y_data_3_small_plot, dtype=np.float32)
mask_3_small = y_data_3_small_plot > 0.385
plot_combs_3_small = np.array(comb_ref_3_small, dtype=np.int8)[mask_3_small]

fig_3 = plt.figure(figsize=(14, 6))

ax_3_big = fig_3.add_subplot(1, 2, 1, projection="3d")
ax_3_big.plot_surface(x, y, z, alpha=ALPHA)
ax_3_big.set_title("Лучшие тройки до сокращения сетки")
ax_3_big.view_init(elev=0, azim=0, roll=0)

ax_3_small = fig_3.add_subplot(1, 2, 2, projection="3d")
ax_3_small.plot_surface(x, y, z, alpha=ALPHA)
ax_3_small.set_title("Лучшие тройки после сокращения сетки")
ax_3_small.view_init(elev=0, azim=0, roll=0)

colors_3 = ["green", "green", "blue"]

initial_idx_big = 0
phi_3_big, r_3_big, z_3_big = points_cyl_big[plot_combs_3_big[initial_idx_big]].T
x_3_big, y_3_big, z_3_big = pol2cart(r_3_big, phi_3_big, z_3_big)
scatter_3_big = ax_3_big.scatter(x_3_big, y_3_big, z_3_big, s=10, color=colors_3)

initial_idx_small = 0
phi_3_small, r_3_small, z_3_small = points_cyl_small[
    plot_combs_3_small[initial_idx_small]
].T
x_3_small, y_3_small, z_3_small = pol2cart(r_3_small, phi_3_small, z_3_small)
scatter_3_small = ax_3_small.scatter(
    x_3_small, y_3_small, z_3_small, s=10, color=colors_3
)

ax_slider_3_big = plt.axes([0.12, 0.00, 0.35, 0.03])
ax_slider_3_small = plt.axes([0.58, 0.00, 0.35, 0.03])

slider_3_big = Slider(
    ax=ax_slider_3_big,
    label="Big comb №",
    valmin=0,
    valmax=len(plot_combs_3_big) - 1,
    valinit=initial_idx_big,
    valstep=1,
)

slider_3_small = Slider(
    ax=ax_slider_3_small,
    label="Small comb №",
    valmin=0,
    valmax=len(plot_combs_3_small) - 1,
    valinit=initial_idx_small,
    valstep=1,
)


def update_big(val):
    global scatter_3_big
    scatter_3_big.remove()
    idx = int(slider_3_big.val)
    phi_temp, r_temp, z_temp = points_cyl_big[plot_combs_3_big[idx]].T
    x_temp, y_temp, z_temp = pol2cart(r_temp, phi_temp, z_temp)
    scatter_3_big = ax_3_big.scatter(x_temp, y_temp, z_temp, s=10, color=colors_3)
    fig_3.canvas.draw_idle()


def update_small(val):
    global scatter_3_small
    scatter_3_small.remove()
    idx = int(slider_3_small.val)
    phi_temp, r_temp, z_temp = points_cyl_small[plot_combs_3_small[idx]].T
    x_temp, y_temp, z_temp = pol2cart(r_temp, phi_temp, z_temp)
    scatter_3_small = ax_3_small.scatter(x_temp, y_temp, z_temp, s=10, color=colors_3)
    fig_3.canvas.draw_idle()


slider_3_big.on_changed(update_big)
slider_3_small.on_changed(update_small)


plt.tight_layout()
plt.show()

## 4 electrodes (3 active, 1 reference)

In [16]:
y_data_4_big_plot = []
comb_ref_4_big = []
for i in range(len(y_data_4_big)):
    y_data_4_big_plot.append(y_data_4_big[i][0].item())
    comb_ref_4_big.append(y_data_4_big[i][1])
y_data_4_big_plot = np.array(y_data_4_big_plot, dtype=np.float32)
mask_4_big = y_data_4_big_plot > 0.494
plot_combs_4_big = np.array(comb_ref_4_big, dtype=np.int8)[mask_4_big]

y_data_4_small_plot = []
comb_ref_4_small = []
for i in range(len(y_data_4_small)):
    y_data_4_small_plot.append(y_data_4_small[i][0].item())
    comb_ref_4_small.append(y_data_4_small[i][1])
y_data_4_small_plot = np.array(y_data_4_small_plot, dtype=np.float32)
mask_4_small = y_data_4_small_plot > 0.465
plot_combs_4_small = np.array(comb_ref_4_small, dtype=np.int8)[mask_4_small]

fig_3 = plt.figure(figsize=(14, 6))

ax_4_big = fig_3.add_subplot(1, 2, 1, projection="3d")
ax_4_big.plot_surface(x, y, z, alpha=ALPHA)
ax_4_big.set_title("Лучшие четвёрки до сокращения сетки")
ax_4_big.view_init(elev=0, azim=0, roll=0)

ax_4_small = fig_3.add_subplot(1, 2, 2, projection="3d")
ax_4_small.plot_surface(x, y, z, alpha=ALPHA)
ax_4_small.set_title("Лучшие четвёрки после сокращения сетки")
ax_4_small.view_init(elev=0, azim=0, roll=0)

colors_4 = ["green", "green", "green", "blue"]

initial_idx_big = 0
phi_4_big, r_4_big, z_4_big = points_cyl_big[plot_combs_4_big[initial_idx_big]].T
x_4_big, y_4_big, z_4_big = pol2cart(r_4_big, phi_4_big, z_4_big)
scatter_4_big = ax_4_big.scatter(x_4_big, y_4_big, z_4_big, s=10, color=colors_4)

initial_idx_small = 0
phi_4_small, r_4_small, z_4_small = points_cyl_small[
    plot_combs_4_small[initial_idx_small]
].T
x_4_small, y_4_small, z_4_small = pol2cart(r_4_small, phi_4_small, z_4_small)
scatter_4_small = ax_4_small.scatter(
    x_4_small, y_4_small, z_4_small, s=10, color=colors_4
)

ax_slider_4_big = plt.axes([0.12, 0.00, 0.35, 0.03])
ax_slider_4_small = plt.axes([0.58, 0.00, 0.35, 0.03])

slider_4_big = Slider(
    ax=ax_slider_4_big,
    label="Big comb №",
    valmin=0,
    valmax=len(plot_combs_4_big) - 1,
    valinit=initial_idx_big,
    valstep=1,
)

slider_4_small = Slider(
    ax=ax_slider_4_small,
    label="Small comb №",
    valmin=0,
    valmax=len(plot_combs_4_small) - 1,
    valinit=initial_idx_small,
    valstep=1,
)


def update_big(val):
    global scatter_4_big
    scatter_4_big.remove()
    idx = int(slider_4_big.val)
    phi_temp, r_temp, z_temp = points_cyl_big[plot_combs_4_big[idx]].T
    x_temp, y_temp, z_temp = pol2cart(r_temp, phi_temp, z_temp)
    scatter_4_big = ax_4_big.scatter(x_temp, y_temp, z_temp, s=10, color=colors_4)
    fig_3.canvas.draw_idle()


def update_small(val):
    global scatter_4_small
    scatter_4_small.remove()
    idx = int(slider_4_small.val)
    phi_temp, r_temp, z_temp = points_cyl_small[plot_combs_4_small[idx]].T
    x_temp, y_temp, z_temp = pol2cart(r_temp, phi_temp, z_temp)
    scatter_4_small = ax_4_small.scatter(x_temp, y_temp, z_temp, s=10, color=colors_4)
    fig_3.canvas.draw_idle()


slider_4_big.on_changed(update_big)
slider_4_small.on_changed(update_small)


plt.tight_layout()
plt.show()

## 5 electrodes (4 active, 1 reference)

In [20]:
y_data_5_big_plot = []
comb_ref_5_big = []
for i in range(len(y_data_5_big)):
    y_data_5_big_plot.append(y_data_5_big[i][0].item())
    comb_ref_5_big.append(y_data_5_big[i][1])
y_data_5_big_plot = np.array(y_data_5_big_plot, dtype=np.float32)
mask_5_big = y_data_5_big_plot > 0.538
plot_combs_5_big = np.array(comb_ref_5_big, dtype=np.int8)[mask_5_big]

y_data_5_small_plot = []
comb_ref_5_small = []
for i in range(len(y_data_5_small)):
    y_data_5_small_plot.append(y_data_5_small[i][0].item())
    comb_ref_5_small.append(y_data_5_small[i][1])
y_data_5_small_plot = np.array(y_data_5_small_plot, dtype=np.float32)
mask_5_small = y_data_5_small_plot > 0.523
plot_combs_5_small = np.array(comb_ref_5_small, dtype=np.int8)[mask_5_small]

fig_3 = plt.figure(figsize=(14, 6))

ax_5_big = fig_3.add_subplot(1, 2, 1, projection="3d")
ax_5_big.plot_surface(x, y, z, alpha=ALPHA)
ax_5_big.set_title("Лучшие пятёрки до сокращения сетки")
ax_5_big.view_init(elev=0, azim=0, roll=0)

ax_5_small = fig_3.add_subplot(1, 2, 2, projection="3d")
ax_5_small.plot_surface(x, y, z, alpha=ALPHA)
ax_5_small.set_title("Лучшие пятёрки после сокращения сетки")
ax_5_small.view_init(elev=0, azim=0, roll=0)

colors_5 = ["green", "green", "green", "green", "blue"]

initial_idx_big = 0
phi_5_big, r_5_big, z_5_big = points_cyl_big[plot_combs_5_big[initial_idx_big]].T
x_5_big, y_5_big, z_5_big = pol2cart(r_5_big, phi_5_big, z_5_big)
scatter_5_big = ax_5_big.scatter(x_5_big, y_5_big, z_5_big, s=10, color=colors_5)

initial_idx_small = 0
phi_5_small, r_5_small, z_5_small = points_cyl_small[
    plot_combs_5_small[initial_idx_small]
].T
x_5_small, y_5_small, z_5_small = pol2cart(r_5_small, phi_5_small, z_5_small)
scatter_5_small = ax_5_small.scatter(
    x_5_small, y_5_small, z_5_small, s=10, color=colors_5
)

ax_slider_5_big = plt.axes([0.12, 0.00, 0.35, 0.03])
ax_slider_5_small = plt.axes([0.58, 0.00, 0.35, 0.03])

slider_5_big = Slider(
    ax=ax_slider_5_big,
    label="Big comb №",
    valmin=0,
    valmax=len(plot_combs_5_big) - 1,
    valinit=initial_idx_big,
    valstep=1,
)

slider_5_small = Slider(
    ax=ax_slider_5_small,
    label="Small comb №",
    valmin=0,
    valmax=len(plot_combs_5_small) - 1,
    valinit=initial_idx_small,
    valstep=1,
)


def update_big(val):
    global scatter_5_big
    scatter_5_big.remove()
    idx = int(slider_5_big.val)
    phi_temp, r_temp, z_temp = points_cyl_big[plot_combs_5_big[idx]].T
    x_temp, y_temp, z_temp = pol2cart(r_temp, phi_temp, z_temp)
    scatter_5_big = ax_5_big.scatter(x_temp, y_temp, z_temp, s=10, color=colors_5)
    fig_3.canvas.draw_idle()


def update_small(val):
    global scatter_5_small
    scatter_5_small.remove()
    idx = int(slider_5_small.val)
    phi_temp, r_temp, z_temp = points_cyl_small[plot_combs_5_small[idx]].T
    x_temp, y_temp, z_temp = pol2cart(r_temp, phi_temp, z_temp)
    scatter_5_small = ax_5_small.scatter(x_temp, y_temp, z_temp, s=10, color=colors_5)
    fig_3.canvas.draw_idle()


slider_5_big.on_changed(update_big)
slider_5_small.on_changed(update_small)


plt.tight_layout()
plt.show()

## 6 electrodes (5 active, 1 reference)

In [26]:
y_data_6_big_plot = []
comb_ref_6_big = []
for i in range(len(y_data_6_big)):
    y_data_6_big_plot.append(y_data_6_big[i][0].item())
    comb_ref_6_big.append(y_data_6_big[i][1])
y_data_6_big_plot = np.array(y_data_6_big_plot, dtype=np.float32)
mask_6_big = y_data_6_big_plot > 0.571
plot_combs_6_big = np.array(comb_ref_6_big, dtype=np.int8)[mask_6_big]

y_data_6_small_plot = []
comb_ref_6_small = []
for i in range(len(y_data_6_small)):
    y_data_6_small_plot.append(y_data_6_small[i][0].item())
    comb_ref_6_small.append(y_data_6_small[i][1])
y_data_6_small_plot = np.array(y_data_6_small_plot, dtype=np.float32)
mask_6_small = y_data_6_small_plot > 0.558
plot_combs_6_small = np.array(comb_ref_6_small, dtype=np.int8)[mask_6_small]

fig_3 = plt.figure(figsize=(14, 6))

ax_6_big = fig_3.add_subplot(1, 2, 1, projection="3d")
ax_6_big.plot_surface(x, y, z, alpha=ALPHA)
ax_6_big.set_title("Лучшие шестёрки до сокращения сетки")
ax_6_big.view_init(elev=0, azim=0, roll=0)

ax_6_small = fig_3.add_subplot(1, 2, 2, projection="3d")
ax_6_small.plot_surface(x, y, z, alpha=ALPHA)
ax_6_small.set_title("Лучшие шестёрки после сокращения сетки")
ax_6_small.view_init(elev=0, azim=0, roll=0)

colors_6 = ["green", "green", "green", "green", "green", "blue"]

initial_idx_big = 0
phi_6_big, r_6_big, z_6_big = points_cyl_big[plot_combs_6_big[initial_idx_big]].T
x_6_big, y_6_big, z_6_big = pol2cart(r_6_big, phi_6_big, z_6_big)
scatter_6_big = ax_6_big.scatter(x_6_big, y_6_big, z_6_big, s=10, color=colors_6)

initial_idx_small = 0
phi_6_small, r_6_small, z_6_small = points_cyl_small[
    plot_combs_6_small[initial_idx_small]
].T
x_6_small, y_6_small, z_6_small = pol2cart(r_6_small, phi_6_small, z_6_small)
scatter_6_small = ax_6_small.scatter(
    x_6_small, y_6_small, z_6_small, s=10, color=colors_6
)

ax_slider_6_big = plt.axes([0.12, 0.00, 0.35, 0.03])
ax_slider_6_small = plt.axes([0.58, 0.00, 0.35, 0.03])

slider_6_big = Slider(
    ax=ax_slider_6_big,
    label="Big comb №",
    valmin=0,
    valmax=len(plot_combs_6_big) - 1,
    valinit=initial_idx_big,
    valstep=1,
)

slider_6_small = Slider(
    ax=ax_slider_6_small,
    label="Small comb №",
    valmin=0,
    valmax=len(plot_combs_6_small) - 1,
    valinit=initial_idx_small,
    valstep=1,
)


def update_big(val):
    global scatter_6_big
    scatter_6_big.remove()
    idx = int(slider_6_big.val)
    phi_temp, r_temp, z_temp = points_cyl_big[plot_combs_6_big[idx]].T
    x_temp, y_temp, z_temp = pol2cart(r_temp, phi_temp, z_temp)
    scatter_6_big = ax_6_big.scatter(x_temp, y_temp, z_temp, s=10, color=colors_6)
    fig_3.canvas.draw_idle()


def update_small(val):
    global scatter_6_small
    scatter_6_small.remove()
    idx = int(slider_6_small.val)
    phi_temp, r_temp, z_temp = points_cyl_small[plot_combs_6_small[idx]].T
    x_temp, y_temp, z_temp = pol2cart(r_temp, phi_temp, z_temp)
    scatter_6_small = ax_6_small.scatter(x_temp, y_temp, z_temp, s=10, color=colors_6)
    fig_3.canvas.draw_idle()


slider_6_big.on_changed(update_big)
slider_6_small.on_changed(update_small)


plt.tight_layout()
plt.show()

## 7 electrodes (6 active, 1 reference)

In [22]:
y_data_7_small_plot = []
comb_ref_7_small = []
for i in range(len(y_data_7_small)):
    y_data_7_small_plot.append(y_data_7_small[i][0].item())
    comb_ref_7_small.append(y_data_7_small[i][1])

y_data_7_small_plot = np.array(y_data_7_small_plot, dtype=np.float32)
mask_7_small = y_data_7_small_plot > 0.587
plot_combs_7_small = np.array(comb_ref_7_small, dtype=np.int8)[mask_7_small]

fig_3 = plt.figure(figsize=(12, 6))

ax_7_small = fig_3.add_subplot(111, projection="3d")
ax_7_small.plot_surface(x, y, z, alpha=ALPHA)
ax_7_small.set_title("Лучшие семёрки после сокращения сетки")
ax_7_small.view_init(elev=0, azim=0, roll=0)

colors_7 = ["green", "green", "green", "green", "green", "green", "blue"]

initial_idx_small = 0
phi_7_small, r_7_small, z_7_small = points_cyl_small[
    plot_combs_7_small[initial_idx_small]
].T
x_7_small, y_7_small, z_7_small = pol2cart(r_7_small, phi_7_small, z_7_small)
scatter_7_small = ax_7_small.scatter(
    x_7_small, y_7_small, z_7_small, s=10, color=colors_7
)

ax_slider_7_small = plt.axes([0.35, 0.00, 0.35, 0.03])


slider_7_small = Slider(
    ax=ax_slider_7_small,
    label="Small comb №",
    valmin=0,
    valmax=len(plot_combs_7_small) - 1,
    valinit=initial_idx_small,
    valstep=1,
)


def update_small(val):
    global scatter_7_small
    scatter_7_small.remove()
    idx = int(slider_7_small.val)
    phi_temp, r_temp, z_temp = points_cyl_small[plot_combs_7_small[idx]].T
    x_temp, y_temp, z_temp = pol2cart(r_temp, phi_temp, z_temp)
    scatter_7_small = ax_7_small.scatter(x_temp, y_temp, z_temp, s=10, color=colors_7)
    fig_3.canvas.draw_idle()


slider_7_small.on_changed(update_small)


plt.tight_layout()
plt.show()

## 8 electrodes (7 active, 1 reference)

In [23]:
y_data_8_small_plot = []
comb_ref_8_small = []
for i in range(len(y_data_8_small)):
    y_data_8_small_plot.append(y_data_8_small[i][0].item())
    comb_ref_8_small.append(y_data_8_small[i][1])

y_data_8_small_plot = np.array(y_data_8_small_plot, dtype=np.float32)
mask_8_small = y_data_8_small_plot > 0.613
plot_combs_8_small = np.array(comb_ref_8_small, dtype=np.int8)[mask_8_small]

fig_3 = plt.figure(figsize=(12, 6))

ax_8_small = fig_3.add_subplot(111, projection="3d")
ax_8_small.plot_surface(x, y, z, alpha=ALPHA)
ax_8_small.set_title("Лучшие восьмёрки после сокращения сетки")
ax_8_small.view_init(elev=0, azim=0, roll=0)

colors_8 = ["green", "green", "green", "green", "green", "green", "green", "blue"]

initial_idx_small = 0
phi_8_small, r_8_small, z_8_small = points_cyl_small[
    plot_combs_8_small[initial_idx_small]
].T
x_8_small, y_8_small, z_8_small = pol2cart(r_8_small, phi_8_small, z_8_small)
scatter_8_small = ax_8_small.scatter(
    x_8_small, y_8_small, z_8_small, s=10, color=colors_8
)

ax_slider_8_small = plt.axes([0.35, 0.00, 0.35, 0.03])


slider_8_small = Slider(
    ax=ax_slider_8_small,
    label="Small comb №",
    valmin=0,
    valmax=len(plot_combs_8_small) - 1,
    valinit=initial_idx_small,
    valstep=1,
)


def update_small(val):
    global scatter_8_small
    scatter_8_small.remove()
    idx = int(slider_8_small.val)
    phi_temp, r_temp, z_temp = points_cyl_small[plot_combs_8_small[idx]].T
    x_temp, y_temp, z_temp = pol2cart(r_temp, phi_temp, z_temp)
    scatter_8_small = ax_8_small.scatter(x_temp, y_temp, z_temp, s=10, color=colors_8)
    fig_3.canvas.draw_idle()


slider_8_small.on_changed(update_small)


plt.tight_layout()
plt.show()

# Metric distributions

In [24]:
alpha_3_big, beta_par_3_big = 9.067, 20.121
x_grid_3_big = np.linspace(0, np.max(y_data_3_big_plot), 500)
pdf_3_big = beta.pdf(x_grid_3_big, alpha_3_big, beta_par_3_big)

alpha_4_big, beta_par_4_big = 18.067, 28.121
x_grid_4_big = np.linspace(0, np.max(y_data_4_big_plot), 500)
pdf_4_big = beta.pdf(x_grid_4_big, alpha_4_big, beta_par_4_big)

alpha_5_big, beta_par_5_big = 25.067, 32.121
x_grid_5_big = np.linspace(0, np.max(y_data_5_big_plot), 500)
pdf_5_big = beta.pdf(x_grid_5_big, alpha_5_big, beta_par_5_big)

alpha_6_big, beta_par_6_big = 34.5, 36.8
x_grid_6_big = np.linspace(0, np.max(y_data_6_big_plot), 500)
pdf_6_big = beta.pdf(x_grid_6_big, alpha_6_big, beta_par_6_big)

mask_3_big = pdf_3_big >= 1
mask_4_big = pdf_4_big >= 1
mask_5_big = pdf_5_big >= 1
mask_6_big = pdf_6_big >= 1

fig_hist_big, ax_hist_big = plt.subplots(figsize=(9, 5))
fig_hist_big.suptitle(
    "Распределение метрики для электродных конфигураций до сокращения сетки"
)

ax_hist_big.plot(
    x_grid_3_big[mask_3_big], pdf_3_big[mask_3_big], "k-", linewidth=2, label="3"
)
ax_hist_big.plot(
    x_grid_4_big[mask_4_big], pdf_4_big[mask_4_big], "g-", linewidth=2, label="4"
)
ax_hist_big.plot(
    x_grid_5_big[mask_5_big], pdf_5_big[mask_5_big], "r-", linewidth=2, label="5"
)
ax_hist_big.plot(
    x_grid_6_big[mask_6_big], pdf_6_big[mask_6_big], "b-", linewidth=2, label="6"
)
fig_hist_big.legend()

In [25]:
alpha_3_small, beta_par_3_small, _, _ = beta.fit(y_data_3_small_plot, floc=0, fscale=1)
x_grid_3_small = np.linspace(0, np.max(y_data_3_small_plot), 500)
pdf_3_small = beta.pdf(x_grid_3_small, alpha_3_small, beta_par_3_small)

alpha_4_small, beta_par_4_small, _, _ = beta.fit(y_data_4_small_plot, floc=0, fscale=1)
x_grid_4_small = np.linspace(0, np.max(y_data_4_small_plot), 500)
pdf_4_small = beta.pdf(x_grid_4_small, alpha_4_small, beta_par_4_small)

alpha_5_small, beta_par_5_small, _, _ = beta.fit(y_data_5_small_plot, floc=0, fscale=1)
x_grid_5_small = np.linspace(0, np.max(y_data_5_small_plot), 500)
pdf_5_small = beta.pdf(x_grid_5_small, alpha_5_small, beta_par_5_small)

alpha_6_small, beta_par_6_small, _, _ = beta.fit(y_data_6_small_plot, floc=0, fscale=1)
x_grid_6_small = np.linspace(0, np.max(y_data_6_small_plot), 500)
pdf_6_small = beta.pdf(x_grid_6_small, alpha_6_small, beta_par_6_small)

alpha_7_small, beta_par_7_small, _, _ = beta.fit(y_data_7_small_plot, floc=0, fscale=1)
x_grid_7_small = np.linspace(0, np.max(y_data_7_small_plot), 500)
pdf_7_small = beta.pdf(x_grid_7_small, alpha_7_small, beta_par_7_small)

alpha_8_small, beta_par_8_small, _, _ = beta.fit(y_data_8_small_plot, floc=0, fscale=1)
x_grid_8_small = np.linspace(0, np.max(y_data_8_small_plot), 500)
pdf_8_small = beta.pdf(x_grid_8_small, alpha_8_small, beta_par_8_small)

mask_3_small = pdf_3_small >= 1
mask_4_small = pdf_4_small >= 1
mask_5_small = pdf_5_small >= 1
mask_6_small = pdf_6_small >= 1
mask_7_small = pdf_7_small >= 1
mask_8_small = pdf_8_small >= 1

fig_hist_small, ax_hist_small = plt.subplots(figsize=(9, 5))
fig_hist_small.suptitle(
    "Распределение метрики для электродных конфигураций после сокращения сетки"
)

ax_hist_small.plot(
    x_grid_3_small[mask_3_small],
    pdf_3_small[mask_3_small],
    "k-",
    linewidth=2,
    label="3",
)
ax_hist_small.plot(
    x_grid_4_small[mask_4_small],
    pdf_4_small[mask_4_small],
    "g-",
    linewidth=2,
    label="4",
)
ax_hist_small.plot(
    x_grid_5_small[mask_5_small],
    pdf_5_small[mask_5_small],
    "r-",
    linewidth=2,
    label="5",
)
ax_hist_small.plot(
    x_grid_6_small[mask_6_small],
    pdf_6_small[mask_6_small],
    "b-",
    linewidth=2,
    label="6",
)
ax_hist_small.plot(
    x_grid_7_small[mask_7_small],
    pdf_7_small[mask_7_small],
    "y-",
    linewidth=2,
    label="7",
)
ax_hist_small.plot(
    x_grid_8_small[mask_8_small],
    pdf_8_small[mask_8_small],
    "c-",
    linewidth=2,
    label="8",
)
fig_hist_small.legend()